# 01 — Preprocessing & EDA (Beijing Multi-Site Air Quality)
Mục tiêu: tải dữ liệu, làm sạch, tạo nhãn phân lớp (AQI class theo PM2.5 24h mean), tạo đặc trưng thời gian + lag, và lưu `data/processed/cleaned.parquet`.

**Lưu ý:** nếu `USE_UCIMLREPO=True` thì notebook cần internet để tải dataset từ UCI.

In [ ]:
USE_UCIMLREPO = False
RAW_ZIP_PATH = "data/raw/PRSA2017_Data_20130301-20170228.zip"

OUTPUT_CLEANED_PATH = 'data/processed/cleaned.parquet'
LAG_HOURS=[1, 3, 24]


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from src.classification_library import (
    load_beijing_air_quality,
    clean_air_quality_df,
    add_pm25_24h_and_label,
    add_time_features,
    add_lag_features,
)

PROJECT_ROOT = Path('..').resolve()
OUT_PATH = (PROJECT_ROOT / OUTPUT_CLEANED_PATH).resolve()
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
df_raw = load_beijing_air_quality(use_ucimlrepo=USE_UCIMLREPO, raw_zip_path=RAW_ZIP_PATH)
print('raw shape:', df_raw.shape)
df_raw.head()

In [ ]:
df = clean_air_quality_df(df_raw)
df = add_pm25_24h_and_label(df)
df = add_time_features(df)
df = add_lag_features(df, lag_hours=LAG_HOURS)
print('cleaned shape:', df.shape)
df[['datetime','station','PM2.5','pm25_24h','aqi_class']].head(10)

In [ ]:
# EDA nhanh: missingness và phân bố lớp
missing_rate = df.isna().mean().sort_values(ascending=False)
missing_rate.head(20)

In [ ]:
class_dist = df['aqi_class'].value_counts(dropna=False)
class_dist

In [ ]:
import matplotlib.pyplot as plt

class_dist.drop(index=[x for x in class_dist.index if pd.isna(x)], errors='ignore').plot(kind='bar')
plt.title('AQI class distribution (PM2.5 24h mean)')
plt.ylabel('count')
plt.tight_layout()
plt.show()

In [ ]:
df.to_parquet(OUT_PATH, index=False)
print('Saved:', OUT_PATH)

## Q1: Phân tích dữ liệu chuỗi thời gian & kiểm tra các thành phần cần thiết
### 1. Kiểm tra khoảng thời gian dữ liệu phủ và tần suất liên tục
- Kiểm tra start/end, số lượng bản ghi, xác nhận dữ liệu liên tục theo giờ.
### 2. Tỷ lệ thiếu theo từng biến và theo thời gian
- Tính missing rate từng biến, vẽ missing theo thời gian để xem có tập trung vào giai đoạn nào không.
### 3. Boxplot/quantile cho ngoại lai
- Vẽ boxplot, tính các quantile để nhận diện outlier và phân phối lệch.
### 4. Vẽ chuỗi pm2.5 toàn giai đoạn và phóng to 1–2 tháng
- Giúp quan sát xu hướng, mùa vụ, spike.
### 5. Kiểm tra tự tương quan
- So sánh pm2.5 với lag 24h, 168h để gợi ý chu kỳ ngày/tuần.
### 6. Kiểm tra tính dừng: adf/kpss
- Chạy kiểm định và nhận xét ngắn gọn.
### 7. Giải thích biến nào thiếu là đáng lo nhất cho dự báo pm2.5 và vì sao

In [ ]:
# 1. Kiểm tra khoảng thời gian dữ liệu phủ và tần suất liên tục
start = df['datetime'].min()
end = df['datetime'].max()
n_hours = (end - start).total_seconds() / 3600 + 1
print(f"Start: {start}, End: {end}, Số giờ: {n_hours}")
print(f"Số dòng: {len(df)}")
# Kiểm tra có bị thiếu giờ nào không
df_sorted = df.sort_values('datetime')
missing_hours = pd.date_range(start, end, freq='H').difference(df_sorted['datetime'])
print(f"Số giờ bị thiếu: {len(missing_hours)}")

In [ ]:
# 2. Tỷ lệ thiếu theo từng biến và theo thời gian
missing_rate = df.isna().mean().sort_values(ascending=False)
print(missing_rate)
# Vẽ missing theo thời gian cho PM2.5
import matplotlib.pyplot as plt
plt.figure(figsize=(12,3))
plt.plot(df['datetime'], df['PM2.5'].isna(), '.', alpha=0.2)
plt.title('Missing PM2.5 over time')
plt.xlabel('Datetime')
plt.ylabel('Missing (True/False)')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Boxplot/quantile cho ngoại lai
plt.figure(figsize=(8,4))
df['PM2.5'].plot(kind='box')
plt.title('Boxplot PM2.5')
plt.show()
print(df['PM2.5'].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))

In [ ]:
# 4. Vẽ chuỗi pm2.5 toàn giai đoạn và phóng to 1–2 tháng
plt.figure(figsize=(14,4))
plt.plot(df['datetime'], df['PM2.5'], alpha=0.5)
plt.title('PM2.5 over entire period')
plt.xlabel('Datetime')
plt.ylabel('PM2.5')
plt.tight_layout()
plt.show()
# Zoom 2 tháng đầu
df_zoom = df[(df['datetime']>=df['datetime'].min()) & (df['datetime']<=df['datetime'].min()+pd.Timedelta(days=60))]
plt.figure(figsize=(14,4))
plt.plot(df_zoom['datetime'], df_zoom['PM2.5'], alpha=0.7)
plt.title('PM2.5 (first 2 months)')
plt.xlabel('Datetime')
plt.ylabel('PM2.5')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Kiểm tra tự tương quan: pm2.5 với lag 24h, 168h
df['pm25_lag24'] = df['PM2.5'].shift(24)
df['pm25_lag168'] = df['PM2.5'].shift(168)
corr_24 = df[['PM2.5','pm25_lag24']].corr().iloc[0,1]
corr_168 = df[['PM2.5','pm25_lag168']].corr().iloc[0,1]
print(f'Tương quan PM2.5 với lag 24h: {corr_24:.3f}')
print(f'Tương quan PM2.5 với lag 168h: {corr_168:.3f}')

In [ ]:
# 6. Kiểm tra tính dừng: adf/kpss
from statsmodels.tsa.stattools import adfuller, kpss
pm25_series = df['PM2.5'].dropna()
adf_result = adfuller(pm25_series)
print(f"ADF Statistic: {adf_result[0]:.3f}, p-value: {adf_result[1]:.3g}")
kpss_result = kpss(pm25_series, nlags='auto')
print(f"KPSS Statistic: {kpss_result[0]:.3f}, p-value: {kpss_result[1]:.3g}")
# Nhận xét ngắn gọn:
if adf_result[1] < 0.05:
    print('Chuỗi có xu hướng dừng theo ADF.')
else:
    print('Chuỗi chưa dừng theo ADF.')
if kpss_result[1] > 0.05:
    print('Chuỗi phù hợp giả định dừng theo KPSS.')
else:
    print('Chuỗi chưa dừng theo KPSS.')

### 7. Nhận xét: Biến nào thiếu là đáng lo nhất cho dự báo pm2.5?
- Biến PM2.5 là mục tiêu dự báo, nếu thiếu sẽ ảnh hưởng trực tiếp đến chất lượng mô hình và đánh giá.
- Các biến khí tượng (temp, pres, dewp, rain, wspm) thường ít thiếu, nhưng nếu thiếu nhiều sẽ làm giảm khả năng giải thích nguyên nhân biến động PM2.5.
- Các biến ô nhiễm khác (co, o3, no2, so2, pm10) nếu thiếu nhiều sẽ làm giảm khả năng phân tích mối liên hệ giữa các chất ô nhiễm.
- Đặc biệt, nếu thiếu dữ liệu PM2.5 ở các giai đoạn cao điểm ô nhiễm (spike), mô hình sẽ khó học được các pattern quan trọng cho cảnh báo sớm.